In [1]:
import pandas as pd

meta = pd.read_csv('meta_data.csv')
meta['얼굴피부타입'].value_counts()

얼굴피부타입
건성      303
중성      260
지성      187
복합지성    161
복합건성    157
심한건성      4
Name: count, dtype: int64

In [2]:
meta['얼굴피부타입'] = meta['얼굴피부타입'].replace({
    '복합지성': '복합성',
    '복합건성': '복합성',
    '심한건성': '건성'
})

print(meta['얼굴피부타입'].value_counts())

얼굴피부타입
복합성    318
건성     307
중성     260
지성     187
Name: count, dtype: int64


In [5]:
meta

,subject_no,성별,나이,얼굴피부타입,자가민감여부,측정날짜
0,1,여성,55,복합성,아니오,2023-07-27
1,2,여성,50,중성,아니오,2023-07-27
2,3,여성,24,중성,아니오,2023-07-27
3,4,여성,47,복합성,예,2023-07-27
4,6,여성,55,복합성,아니오,2023-07-27
...,...,...,...,...,...,...
1067,1096,여성,25,건성,아니오,2023-10-17
1068,1097,여성,24,건성,아니오,2023-10-17
1069,1098,여성,23,건성,아니오,2023-10-17
1070,1099,여성,26,건성,예,2023-10-17


In [3]:
import os
import json
from PIL import Image
import shutil
from tqdm import tqdm  # tqdm 라이브러리 임포트

# 원본 이미지가 있는 폴더 경로
source_folder = './train_스마트폰/'
# 라벨링이 되어있는 JSON 파일이 있는 폴더 경로
label_folder = './train_label/'
# 이미지가 이동될 train 폴더
destination_folder = './train/'

# JSON 파일에서 bbox 정보를 읽어오는 함수
def load_bbox_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        bbox = data['images']['bbox']
    return bbox

# 이미지를 크롭하고 저장하는 함수
def crop_and_save_image(image_path, bbox, save_path):
    with Image.open(image_path) as img:
        cropped_img = img.crop(bbox)  # bbox에 맞춰 이미지 크롭
        cropped_img.save(save_path)   # 크롭한 이미지 저장

# meta 데이터프레임에서 subject_no와 얼굴피부타입을 순회
for idx, row in tqdm(meta.iterrows(), total=meta.shape[0], desc="Processing subjects"):
    subject_no = f"{row['subject_no']:04d}"  # subject_no를 0001 형식으로 변환
    skin_type = row['얼굴피부타입']

    # 원본 이미지 경로 설정
    original_folder = os.path.join(source_folder, subject_no)

    # JSON 라벨이 있는 폴더 경로 설정
    json_label_folder = os.path.join(label_folder, subject_no)
    
    # 타겟 폴더 경로 설정 (예: ./train/지성)
    target_folder = os.path.join(destination_folder, skin_type)

    # 원본 폴더에 해당 subject_no 폴더가 존재하는지 확인
    if os.path.exists(original_folder) and os.path.exists(json_label_folder):
        # 각 이미지에 대해 4개의 크롭된 이미지를 생성 (01, 05, 06, 08)
        for angle in ['F', 'L', 'R']:
            image_filename = f"{subject_no}_03_{angle}.jpg"
            image_path = os.path.join(original_folder, image_filename)
            
            if os.path.exists(image_path):
                # 사용할 맨 뒷자리들 (01, 05, 06, 08)
                for i in [1, 5, 6, 8]:
                    # JSON 파일 경로 설정
                    json_filename = f"{subject_no}_03_{angle}_{i:02d}.json"
                    json_path = os.path.join(json_label_folder, json_filename)
                    
                    if os.path.exists(json_path):
                        # bbox 정보 로드
                        bbox = load_bbox_from_json(json_path)
                        
                        # 크롭된 이미지 저장 경로 설정
                        cropped_image_filename = f"{subject_no}_03_{angle}_{i:02d}.jpg"
                        save_path = os.path.join(target_folder, cropped_image_filename)
                        
                        # 이미지 크롭 및 저장
                        crop_and_save_image(image_path, bbox, save_path)
                        
                        print(f"Cropped and saved {cropped_image_filename} to {target_folder}")
    else:
        print(f"Folder {subject_no} not found in {source_folder} or {json_label_folder}")


Processing subjects:   0%|                                                                    | 0/1072 [00:00<?, ?it/s]

Folder 0001 not found in ./train_스마트폰/ or ./train_label/0001
Cropped and saved 0002_03_F_01.jpg to ./train/중성
Cropped and saved 0002_03_F_05.jpg to ./train/중성
Cropped and saved 0002_03_F_06.jpg to ./train/중성
Cropped and saved 0002_03_F_08.jpg to ./train/중성
Cropped and saved 0002_03_L_01.jpg to ./train/중성
Cropped and saved 0002_03_L_05.jpg to ./train/중성
Cropped and saved 0002_03_L_06.jpg to ./train/중성
Cropped and saved 0002_03_L_08.jpg to ./train/중성
Cropped and saved 0002_03_R_01.jpg to ./train/중성


Processing subjects:   0%|▎                                                           | 5/1072 [00:00<02:09,  8.24it/s]

Cropped and saved 0002_03_R_05.jpg to ./train/중성
Cropped and saved 0002_03_R_06.jpg to ./train/중성
Cropped and saved 0002_03_R_08.jpg to ./train/중성
Cropped and saved 0003_03_F_01.jpg to ./train/중성
Cropped and saved 0003_03_F_05.jpg to ./train/중성
Cropped and saved 0003_03_F_06.jpg to ./train/중성
Cropped and saved 0003_03_F_08.jpg to ./train/중성
Cropped and saved 0003_03_L_01.jpg to ./train/중성
Cropped and saved 0003_03_L_05.jpg to ./train/중성
Cropped and saved 0003_03_L_06.jpg to ./train/중성
Cropped and saved 0003_03_L_08.jpg to ./train/중성
Cropped and saved 0003_03_R_01.jpg to ./train/중성
Cropped and saved 0003_03_R_05.jpg to ./train/중성
Cropped and saved 0003_03_R_06.jpg to ./train/중성
Cropped and saved 0003_03_R_08.jpg to ./train/중성
Folder 0004 not found in ./train_스마트폰/ or ./train_label/0004
Cropped and saved 0006_03_F_01.jpg to ./train/복합성
Cropped and saved 0006_03_F_05.jpg to ./train/복합성
Cropped and saved 0006_03_F_06.jpg to ./train/복합성
Cropped and saved 0006_03_F_08.jpg to ./train/복합성
Crop

Processing subjects:   1%|▍                                                           | 7/1072 [00:01<03:48,  4.67it/s]

...

Cropped and saved 1098_03_R_08.jpg to ./train/건성
Cropped and saved 1099_03_F_01.jpg to ./train/건성
Cropped and saved 1099_03_F_05.jpg to ./train/건성
Cropped and saved 1099_03_F_06.jpg to ./train/건성
Cropped and saved 1099_03_F_08.jpg to ./train/건성
Cropped and saved 1099_03_L_01.jpg to ./train/건성
Cropped and saved 1099_03_L_05.jpg to ./train/건성
Cropped and saved 1099_03_L_06.jpg to ./train/건성
Cropped and saved 1099_03_L_08.jpg to ./train/건성
Cropped and saved 1099_03_R_01.jpg to ./train/건성
Cropped and saved 1099_03_R_05.jpg to ./train/건성
Cropped and saved 1099_03_R_06.jpg to ./train/건성


Processing subjects: 100%|████████████████████████████████████████████████████████▉| 1071/1072 [13:55<00:01,  1.04s/it]

Cropped and saved 1099_03_R_08.jpg to ./train/건성
Cropped and saved 1100_03_F_01.jpg to ./train/복합성
Cropped and saved 1100_03_F_05.jpg to ./train/복합성
Cropped and saved 1100_03_F_06.jpg to ./train/복합성
Cropped and saved 1100_03_F_08.jpg to ./train/복합성
Cropped and saved 1100_03_L_01.jpg to ./train/복합성
Cropped and saved 1100_03_L_05.jpg to ./train/복합성
Cropped and saved 1100_03_L_06.jpg to ./train/복합성
Cropped and saved 1100_03_L_08.jpg to ./train/복합성
Cropped and saved 1100_03_R_01.jpg to ./train/복합성
Cropped and saved 1100_03_R_05.jpg to ./train/복합성
Cropped and saved 1100_03_R_06.jpg to ./train/복합성


Processing subjects: 100%|█████████████████████████████████████████████████████████| 1072/1072 [13:56<00:00,  1.28it/s]

Cropped and saved 1100_03_R_08.jpg to ./train/복합성


In [4]:
import os
import json
from PIL import Image
import shutil
from tqdm import tqdm  # tqdm 라이브러리 임포트

# 원본 이미지가 있는 폴더 경로
source_folder = './vaild_스마트폰/'
# 라벨링이 되어있는 JSON 파일이 있는 폴더 경로
label_folder = './vaild_label/'
# 이미지가 이동될 train 폴더
destination_folder = './vaild/'

# JSON 파일에서 bbox 정보를 읽어오는 함수
def load_bbox_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        bbox = data['images']['bbox']
    return bbox

# 이미지를 크롭하고 저장하는 함수
def crop_and_save_image(image_path, bbox, save_path):
    with Image.open(image_path) as img:
        cropped_img = img.crop(bbox)  # bbox에 맞춰 이미지 크롭
        cropped_img.save(save_path)   # 크롭한 이미지 저장

# meta 데이터프레임에서 subject_no와 얼굴피부타입을 순회
for idx, row in tqdm(meta.iterrows(), total=meta.shape[0], desc="Processing subjects"):
    subject_no = f"{row['subject_no']:04d}"  # subject_no를 0001 형식으로 변환
    skin_type = row['얼굴피부타입']

    # 원본 이미지 경로 설정
    original_folder = os.path.join(source_folder, subject_no)
    # JSON 라벨이 있는 폴더 경로 설정
    json_label_folder = os.path.join(label_folder, subject_no)
    
    # 타겟 폴더 경로 설정 (예: ./train/지성)
    target_folder = os.path.join(destination_folder, skin_type)

    # 원본 폴더에 해당 subject_no 폴더가 존재하는지 확인
    if os.path.exists(original_folder) and os.path.exists(json_label_folder):
        # 각 이미지에 대해 4개의 크롭된 이미지를 생성 (01, 05, 06, 08)
        for angle in ['F', 'L', 'R']:
            image_filename = f"{subject_no}_03_{angle}.jpg"
            image_path = os.path.join(original_folder, image_filename)
            
            if os.path.exists(image_path):
                # 사용할 맨 뒷자리들 (01, 05, 06, 08)
                for i in [1, 5, 6, 8]:
                    # JSON 파일 경로 설정
                    json_filename = f"{subject_no}_03_{angle}_{i:02d}.json"
                    json_path = os.path.join(json_label_folder, json_filename)
                    
                    if os.path.exists(json_path):
                        # bbox 정보 로드
                        bbox = load_bbox_from_json(json_path)
                        
                        # 크롭된 이미지 저장 경로 설정
                        cropped_image_filename = f"{subject_no}_03_{angle}_{i:02d}.jpg"
                        save_path = os.path.join(target_folder, cropped_image_filename)
                        
                        # 이미지 크롭 및 저장
                        crop_and_save_image(image_path, bbox, save_path)
                        
                        print(f"Cropped and saved {cropped_image_filename} to {target_folder}")
    else:
        print(f"Folder {subject_no} not found in {source_folder} or {json_label_folder}")


Processing subjects:   0%|                                                                    | 0/1072 [00:00<?, ?it/s]

Cropped and saved 0001_03_F_01.jpg to ./vaild/복합성
Cropped and saved 0001_03_F_05.jpg to ./vaild/복합성
Cropped and saved 0001_03_F_06.jpg to ./vaild/복합성
Cropped and saved 0001_03_F_08.jpg to ./vaild/복합성
Cropped and saved 0001_03_L_01.jpg to ./vaild/복합성
Cropped and saved 0001_03_L_05.jpg to ./vaild/복합성
Cropped and saved 0001_03_L_06.jpg to ./vaild/복합성
Cropped and saved 0001_03_L_08.jpg to ./vaild/복합성


Processing subjects:   0%|                                                            | 1/1072 [00:00<12:09,  1.47it/s]

Cropped and saved 0001_03_R_01.jpg to ./vaild/복합성
Cropped and saved 0001_03_R_05.jpg to ./vaild/복합성
Cropped and saved 0001_03_R_06.jpg to ./vaild/복합성
Cropped and saved 0001_03_R_08.jpg to ./vaild/복합성
Folder 0002 not found in ./vaild_스마트폰/ or ./vaild_label/0002
Folder 0003 not found in ./vaild_스마트폰/ or ./vaild_label/0003
Cropped and saved 0004_03_F_01.jpg to ./vaild/복합성
Cropped and saved 0004_03_F_05.jpg to ./vaild/복합성


Processing subjects:   0%|▏                                                           | 4/1072 [00:00<03:01,  5.87it/s]

...

Cropped and saved 1076_03_R_08.jpg to ./vaild/지성
Folder 1077 not found in ./vaild_스마트폰/ or ./vaild_label/1077
Folder 1078 not found in ./vaild_스마트폰/ or ./vaild_label/1078
Folder 1079 not found in ./vaild_스마트폰/ or ./vaild_label/1079
Folder 1080 not found in ./vaild_스마트폰/ or ./vaild_label/1080
Cropped and saved 1081_03_F_01.jpg to ./vaild/지성
Cropped and saved 1081_03_F_05.jpg to ./vaild/지성
Cropped and saved 1081_03_F_06.jpg to ./vaild/지성
Cropped and saved 1081_03_F_08.jpg to ./vaild/지성
Cropped and saved 1081_03_L_01.jpg to ./vaild/지성
Cropped and saved 1081_03_L_05.jpg to ./vaild/지성
Cropped and saved 1081_03_L_06.jpg to ./vaild/지성
Cropped and saved 1081_03_L_08.jpg to ./vaild/지성
Cropped and saved 1081_03_R_01.jpg to ./vaild/지성
Cropped and saved 1081_03_R_05.jpg to ./vaild/지성
Cropped and saved 1081_03_R_06.jpg to ./vaild/지성


Processing subjects:  98%|███████████████████████████████████████████████████████▉ | 1053/1072 [01:45<00:03,  5.41it/s]

Cropped and saved 1081_03_R_08.jpg to ./vaild/지성
Folder 1082 not found in ./vaild_스마트폰/ or ./vaild_label/1082
Folder 1083 not found in ./vaild_스마트폰/ or ./vaild_label/1083
Folder 1084 not found in ./vaild_스마트폰/ or ./vaild_label/1084
Folder 1085 not found in ./vaild_스마트폰/ or ./vaild_label/1085
Folder 1086 not found in ./vaild_스마트폰/ or ./vaild_label/1086
Folder 1087 not found in ./vaild_스마트폰/ or ./vaild_label/1087
Folder 1088 not found in ./vaild_스마트폰/ or ./vaild_label/1088
Folder 1089 not found in ./vaild_스마트폰/ or ./vaild_label/1089
Folder 1090 not found in ./vaild_스마트폰/ or ./vaild_label/1090
Cropped and saved 1091_03_F_01.jpg to ./vaild/복합성
Cropped and saved 1091_03_F_05.jpg to ./vaild/복합성
Cropped and saved 1091_03_F_06.jpg to ./vaild/복합성
Cropped and saved 1091_03_F_08.jpg to ./vaild/복합성
Cropped and saved 1091_03_L_01.jpg to ./vaild/복합성
Cropped and saved 1091_03_L_05.jpg to ./vaild/복합성
Cropped and saved 1091_03_L_06.jpg to ./vaild/복합성
Cropped and saved 1091_03_L_08.jpg to ./vaild/복합성
Cr

Processing subjects: 100%|█████████████████████████████████████████████████████████| 1072/1072 [01:46<00:00, 10.06it/s]

Cropped and saved 1091_03_R_08.jpg to ./vaild/복합성
Folder 1092 not found in ./vaild_스마트폰/ or ./vaild_label/1092
Folder 1093 not found in ./vaild_스마트폰/ or ./vaild_label/1093
Folder 1094 not found in ./vaild_스마트폰/ or ./vaild_label/1094
Folder 1095 not found in ./vaild_스마트폰/ or ./vaild_label/1095
Folder 1096 not found in ./vaild_스마트폰/ or ./vaild_label/1096
Folder 1097 not found in ./vaild_스마트폰/ or ./vaild_label/1097
Folder 1098 not found in ./vaild_스마트폰/ or ./vaild_label/1098
Folder 1099 not found in ./vaild_스마트폰/ or ./vaild_label/1099
Folder 1100 not found in ./vaild_스마트폰/ or ./vaild_label/1100
